# Course 4: Mathematics and AI
## Session 5: Category theory for deep learning I

**Goal:** build a small category whose objects are tensor shapes and whose arrows are layers, and check the laws with tests.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every Python cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a library, a GPU or a Lean installation. Lean code in this course was written carefully but was NOT compiled here: check it in Lean before you teach it.

### 1. Objects and arrows
A category has objects, arrows between them, a way to compose arrows whose ends match, and an identity arrow on every object. Here:
- objects are types such as `R^3`,
- an arrow `f: R^3 -> R^2` is a layer,
- composition is 'run one after the other', allowed only when the types match.

In [ ]:
import numpy as np
class Arrow:
    def __init__(self, dom, cod, fn, name): self.dom, self.cod, self.fn, self.name = dom, cod, fn, name
    def __call__(self, x):
        assert x.shape == (self.dom,), f"{self.name} expects R^{self.dom}, got {x.shape}"
        y = self.fn(x); assert y.shape == (self.cod,); return y
    def __rshift__(self, other):                                  # f >> g means 'f then g'
        if self.cod != other.dom: raise TypeError(f"cannot compose {self.name}: R^{self.dom}->R^{self.cod} with {other.name}: R^{other.dom}->R^{other.cod}")
        return Arrow(self.dom, other.cod, lambda x: other.fn(self.fn(x)), f"({self.name} >> {other.name})")
    def __repr__(self): return f"{self.name}: R^{self.dom} -> R^{self.cod}"

def identity(n): return Arrow(n, n, lambda x: x, f"id_{n}")
rng = np.random.default_rng(0)
def linear(m, n, name):
    W, b = rng.normal(size=(n, m)), rng.normal(size=n); return Arrow(m, n, lambda x: W @ x + b, name)
def relu(n): return Arrow(n, n, lambda x: np.maximum(x, 0), f"relu_{n}")

f, g, h = linear(3, 4, "f"), relu(4), linear(4, 2, "h")
net = f >> g >> h; print(net)
try: f >> h >> g
except TypeError as e: print("type error caught:", e)

### 2. The laws
- Identity: `id >> f = f = f >> id`.
- Associativity: `(f >> g) >> h = f >> (g >> h)`.

Two arrows are equal when they give the same output on every input. A test on random inputs cannot prove that, but it catches mistakes.

In [ ]:
def same(a, b, trials=100):
    return a.dom == b.dom and a.cod == b.cod and all(np.allclose(a(x), b(x)) for x in rng.normal(size=(trials, a.dom)))
print("left identity :", same(identity(3) >> f, f))
print("right identity:", same(f >> identity(4), f))
print("associativity :", same((f >> g) >> h, f >> (g >> h)))
assert same(identity(3) >> f, f) and same(f >> identity(4), f) and same((f >> g) >> h, f >> (g >> h))

### 3. A functor: forget the numbers, keep the shapes
A functor maps objects to objects and arrows to arrows, and it must respect composition and identities. 'Shape inference' is a functor from layers to a tiny category where an arrow is just a pair (input size, output size).

In [ ]:
shape = lambda a: (a.dom, a.cod)
def compose_shapes(s, t):
    assert s[1] == t[0]; return (s[0], t[1])
print("F(f >> g) =", shape(f >> g), "| F(f) then F(g) =", compose_shapes(shape(f), shape(g)))
assert shape(f >> g) == compose_shapes(shape(f), shape(g)) and shape(identity(3)) == (3, 3)

This is why a framework can check a model's shapes without running it: the check is the image of your network under a functor.

### 4. A second way to combine: side by side
Networks also run in parallel. `f ⊗ g` takes a pair of inputs (concatenated) and applies f to the first part and g to the second. With this the category becomes monoidal, and the two ways of combining must agree:

`(f1 >> f2) ⊗ (g1 >> g2) = (f1 ⊗ g1) >> (f2 ⊗ g2)`

In [ ]:
def par(a, b):
    return Arrow(a.dom + b.dom, a.cod + b.cod, lambda x: np.concatenate([a.fn(x[:a.dom]), b.fn(x[a.dom:])]), f"({a.name} ⊗ {b.name})")
f1, f2, g1, g2 = linear(2, 3, "f1"), linear(3, 2, "f2"), linear(4, 1, "g1"), linear(1, 5, "g2")
lhs, rhs = par(f1 >> f2, g1 >> g2), par(f1, g1) >> par(f2, g2)
print(lhs); print("interchange law holds:", same(lhs, rhs)); assert same(lhs, rhs)

### 5. Copy and add: residual connections as a diagram
With two more arrows, `copy: R^n -> R^n ⊗ R^n` and `add: R^n ⊗ R^n -> R^n`, a residual block is `copy >> (f ⊗ id) >> add`.

In [ ]:
copy = lambda n: Arrow(n, 2 * n, lambda x: np.concatenate([x, x]), f"copy_{n}")
add = lambda n: Arrow(2 * n, n, lambda x: x[:n] + x[n:], f"add_{n}")
block = linear(4, 4, "f")
residual = copy(4) >> par(block, identity(4)) >> add(4)
x = rng.normal(size=4); print(residual); print("equals f(x) + x:", np.allclose(residual(x), block(x) + x))
assert np.allclose(residual(x), block(x) + x)

### 6. Something that is not a category
If 'composition' silently broadcasts or reshapes when types do not match, associativity or the identity law can fail. Design APIs so that the laws hold; then refactoring a model cannot change what it computes.

In [ ]:
def sloppy(a, b):                                                # 'helpfully' truncates or pads when the sizes differ
    def fn(x):
        y = a.fn(x); y = y[:b.dom] if len(y) >= b.dom else np.pad(y, (0, b.dom - len(y)))
        return b.fn(y)
    return Arrow(a.dom, b.cod, fn, f"sloppy({a.name},{b.name})")
u = linear(3, 5, "u")
print("with sloppy composition, is id_3 a right identity for u: R^3 -> R^5?", same(sloppy(u, identity(3)), u) if u.cod == 3 else "no: the result has type R^3 -> R^3, information was dropped")

### TODO
1. Add a `Dropout` arrow. In what sense does it break 'equal on every input'? What would the arrows have to be for the laws to hold again?
2. Write the functor that sends each arrow to its parameter count, with addition as composition. Check the two functor laws.
3. Draw the string diagram for a two-layer residual network and for a network with a skip connection across both layers. Are they the same arrow?